### 1.Pre-process the JSON String to fix the Data Quality Issues
### 2.Transfrom JSON string to JSON Object
### 3.Write Transformed data to the silver schema

In [0]:
df_order_data = spark.read.table("gizmobox_nara.bronze.v_orders")
display(df_order_data)

In [0]:
%sql
SELECT * FROM gizmobox_nara.bronze.v_orders

## 1.Pre-Processing the Column to fix data Quality Issues

In [0]:
from pyspark.sql import functions as f
df_order_data_issues = (
    df_order_data
    .select(
        "value",
        f.regexp_replace("value", '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "$1"').alias("fixed_value")
    )
)
display(df_order_data_issues)

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tv_orders_fixed AS
SELECT Value,
regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value   
FROM gizmobox_nara.bronze.v_orders

### 2. Transform JSON String to JSON Object
### *Function schema of json  - schema_of_json()
### *Function from json

In [0]:
df_schema = (
    df_order_data_issues.select(
        f.schema_of_json(f.col("fixed_value")).alias("schema")
    )
)
display(df_schema)

In [0]:
%sql
SELECT schema_of_json(fixed_value) AS schema, fixed_value FROM tv_orders_fixed

In [0]:
# df_schema does not contain 'fixed_value', use df_order_data_issues instead
df_order_json_value = (
    df_order_data_issues.select(
        f.from_json(
            "fixed_value",
            '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''
        ).alias("json_value")
    )
)
display(df_order_json_value)

In [0]:
%sql
SELECT from_json(fixed_value,
                 'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>
                ') AS json_value, fixed_value FROM tv_orders_fixed

In [0]:
%sql
DROP TABLE IF EXISTS gizmobox_nara.silver.orders;
CREATE TABLE IF NOT EXISTS gizmobox_nara.silver.orders_json AS
SELECT from_json(fixed_value,'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'
) AS json_value
 FROM tv_orders_fixed


In [0]:
df_order_json_value.writeTo("gizmobox_nara.silver.py_orders_json").createOrReplace()


In [0]:
df = spark.read.table("gizmobox_nara.silver.py_orders_json")
display(df)

In [0]:
%sql
SELECT * FROM gizmobox_nara.silver.orders_json